In [1]:
import sys, os

WORK_DIR = os.path.dirname(os.getcwd())
sys.path.append(WORK_DIR)

import dataio

vid_dataset = dataio.Video(f'{WORK_DIR}/data/mock_videos/mockvideo_feat_data.npy')
vid_dataset = dataio.Video(f'{WORK_DIR}/data/mock_videos/us_feat_data.npy')

In [2]:
vid_dataset.shape

(60, 224, 224)

In [3]:
from torch.utils.data import DataLoader

batch_size = 1
sample_frac = 38e-4

coord_dataset = dataio.Implicit3DWrapper(vid_dataset, 
                                            sidelength=vid_dataset.shape, 
                                            sample_fraction=sample_frac,
                                            )
dataloader = DataLoader(coord_dataset, 
                        shuffle=True, 
                        batch_size=batch_size,
                        pin_memory=True, 
                        num_workers=0)

import modules, utils, loss_functions, training
from functools import partial

model = modules.SingleBVPNet(type="sine", in_features=3, 
                                out_features=vid_dataset.channels,
                            mode='mlp', 
                            hidden_features=1024, 
                            num_hidden_layers=3)
model.cuda()

logging_root = f'{WORK_DIR}/logs'
print("logging_root: ", logging_root)

experiment_name = 'mock_video_feature_test'
experiment_name = 'us_video_feature_test'
root_path = os.path.join(logging_root, experiment_name)

num_epochs = 1000
lr = 1e-4
steps_til_summary = 100
epochs_til_checkpoint = 100



SingleBVPNet(
  (image_downsampling): ImageDownsampling()
  (net): FCBlock(
    (net): MetaSequential(
      (0): MetaSequential(
        (0): BatchLinear(in_features=3, out_features=1024, bias=True)
        (1): Sine()
      )
      (1): MetaSequential(
        (0): BatchLinear(in_features=1024, out_features=1024, bias=True)
        (1): Sine()
      )
      (2): MetaSequential(
        (0): BatchLinear(in_features=1024, out_features=1024, bias=True)
        (1): Sine()
      )
      (3): MetaSequential(
        (0): BatchLinear(in_features=1024, out_features=1024, bias=True)
        (1): Sine()
      )
      (4): MetaSequential(
        (0): BatchLinear(in_features=1024, out_features=384, bias=True)
      )
    )
  )
)
logging_root:  C:\dev\siren/logs


C:\dev\siren\training.py:7: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
from utils import *
from experiment_scripts.train_feat_test import pca

def write_feat_video_summary_pca(vid_dataset, model, model_input, gt, model_output, writer, total_steps, prefix='train_'):

    resolution = vid_dataset.shape
    # print("vid_dataset.shape: ", vid_dataset.shape)
    # print("vid_dataset.channels: ", vid_dataset.channels)
    # input()
    # (n_frames, h, w)
    # frames = [0, 60, 120, 200] # this is only valid for n_frames >= 200
    frames = [int(frm_frac * resolution[0]) for frm_frac in [0, 0.25, 0.5, 0.75]] # temporary fix

    Nslice = 10
    with torch.no_grad():
        coords = [dataio.get_mgrid((1, resolution[1], resolution[2]), dim=3)[None,...].cuda() for f in frames]
        for idx, f in enumerate(frames):
            coords[idx][..., 0] = (f / (resolution[0] - 1) - 0.5) * 2
        coords = torch.cat(coords, dim=0)

        # print("coords.shape: ", coords.shape)
        # print("frames: ", frames)
        # input()

        # output = torch.zeros(coords.shape) 
        # this is just a coincidence when vid_dataset.channels = 3
        # simply because coords are (x, y, t)
        output = torch.zeros(coords.shape[0], coords.shape[1], vid_dataset.channels)
        # so the coords.shape[0] is the number of slices to visualize
        # coords.shape[1] is the number of pixels
        # we need the third dimension to be the number of channels in vid_dataset to match the ground truth
        split = int(coords.shape[1] / Nslice)
        for i in range(Nslice):
            pred = model({'coords':coords[:, i*split:(i+1)*split, :]})['model_out']
            output[:, i*split:(i+1)*split, :] =  pred.cpu()
        # print("pred.shape: ", pred.shape)
        # print("output.shape: ", output.shape)
        # input()

    
    # pred_vid = output.view(len(frames), resolution[1], resolution[2], 3) / 2 + 0.5 # the channel dimension is hard-coded for 3 channels
    pred_vid = output.view(len(frames), resolution[1], resolution[2], vid_dataset.channels) / 2 + 0.5
    pred_vid = torch.clamp(pred_vid, 0, 1)
    gt_vid = torch.from_numpy(vid_dataset.vid[frames, :, :, :])
    psnr = 10*torch.log10(1 / torch.mean((gt_vid - pred_vid)**2))

    pred_vid = pred_vid.permute(0, 3, 1, 2)
    gt_vid = gt_vid.permute(0, 3, 1, 2)

    # pca the video
    all_pred_vid_pca = []
    all_gt_vid_pca = []
    for i in range(pred_vid.shape[0]):
        # print("pred_vid[i, ...].shape: ", pred_vid[i, ...].shape)
        # print("gt_vid[i, ...].shape: ", gt_vid[i, ...].shape)
        [pred_vid_pca, gt_vid_pca], _ = pca([pred_vid[i, ...].unsqueeze(0), gt_vid[i, ...].unsqueeze(0)])

        # print("pred_vid_pca: ", pred_vid_pca)
        # print("gt_vid_pca: ", gt_vid_pca)
        
        all_pred_vid_pca.append(pred_vid_pca[0].unsqueeze(0))
        all_gt_vid_pca.append(gt_vid_pca[0].unsqueeze(0))

    all_pred_vid_pca = torch.cat(all_pred_vid_pca, dim=0)
    all_gt_vid_pca = torch.cat(all_gt_vid_pca, dim=0)
    # print("all_pred_vid_pca.shape: ", all_pred_vid_pca.shape)
    # print("all_gt_vid_pca.shape: ", all_gt_vid_pca.shape)
    

    # output_vs_gt = torch.cat((gt_vid, pred_vid), dim=-2)
    output_vs_gt = torch.cat((all_gt_vid_pca, all_pred_vid_pca), dim=-2)
    writer.add_image(prefix + 'output_vs_gt', make_grid(output_vs_gt, scale_each=False, normalize=True),
                     global_step=total_steps)
    min_max_summary(prefix + 'coords', model_input['coords'], writer, total_steps)
    min_max_summary(prefix + 'pred_vid', pred_vid, writer, total_steps)
    writer.add_scalar(prefix + "psnr", psnr, total_steps)

In [5]:
# Define the loss
loss_fn = partial(loss_functions.image_mse, None)
summary_fn = partial(write_feat_video_summary_pca, vid_dataset)

training.train(model=model, train_dataloader=dataloader, epochs=num_epochs, lr=lr,
            steps_til_summary=steps_til_summary, epochs_til_checkpoint=epochs_til_checkpoint,
            model_dir=root_path, loss_fn=loss_fn, summary_fn=summary_fn) 

The model directory C:\dev\siren/logs\us_video_feature_test exists. Overwrite? (y/n) y


  0%|▍                                                                                | 5/1000 [00:02<06:07,  2.71it/s]

Epoch 0, Total loss 1.206385, iteration time 2.313493


 10%|████████▎                                                                      | 105/1000 [00:07<01:45,  8.52it/s]

Epoch 100, Total loss 0.009573, iteration time 1.662354


...

 80%|███████████████████████████████████████████████████████████████▌               | 805/1000 [00:39<00:23,  8.26it/s]

Epoch 800, Total loss 0.001352, iteration time 1.734398


 90%|███████████████████████████████████████████████████████████████████████▍       | 905/1000 [00:44<00:11,  8.43it/s]

Epoch 900, Total loss 0.001041, iteration time 1.691455


100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:47<00:00, 21.14it/s]


In [7]:
vid_dataset.channels

384